# Bloom Filter Assignment — Hash Quality and False Positives

# Bloom Filter Setup: Core Functions and Data Preparation


In [ ]:
import hashlib, random, string, math
import matplotlib.pyplot as plt

# Helper to generate random strings
def rand_string(n=12):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=n)).encode()

def sha256_int(b: bytes) -> int:
    return int(hashlib.sha256(b).hexdigest(), 16)

# Bloom filter class (simple)
class BloomFilter:
    def __init__(self, m, k, hash_family):
        self.m, self.k, self.hash_family = m, k, hash_family
        self.bits = [0] * m

    def positions(self, key_bytes):
        return [self.hash_family(i, key_bytes) % self.m for i in range(self.k)]

    def add(self, key_bytes):
        for p in self.positions(key_bytes):
            self.bits[p] = 1

    def contains(self, key_bytes):
        return all(self.bits[p] == 1 for p in self.positions(key_bytes))

    def occupancy(self):
        return sum(self.bits) / self.m

# Experiment helpers
def run_experiment(m, k, inserted, queries, hash_family):
    bf = BloomFilter(m, k, hash_family)
    for x in inserted:
        bf.add(x)
    false_pos = sum(1 for q in queries if bf.contains(q))
    return false_pos / len(queries), bf.occupancy()

def theoretical_fpr(m, k, n):
    return (1 - math.e**(-k * n / m)) ** k

# Data generation
def make_insert_set(n):
    items = set(); out = []
    while len(out) < n:
        x = rand_string()
        if x not in items:
            items.add(x); out.append(x)
    return out

def make_random_queries(inserted, n):
    S = set(inserted); out = []
    while len(out) < n:
        q = rand_string()
        if q not in S:
            out.append(q)
    return out

def make_near_queries(inserted, n):
    out = []
    L = len(inserted[0])
    idx = 0
    while len(out) < n:
        base = bytearray(inserted[idx % len(inserted)])
        pos = random.randrange(L)
        base[pos] = random.choice((string.ascii_letters + string.digits).encode())
        y = bytes(base)
        if y != inserted[idx % len(inserted)]:
            out.append(y)
            idx += 1
    return out

# Parameters
m, k = 10000, 7
n_insert, n_query = 2000, 20000

inserted = make_insert_set(n_insert)
rand_Q = make_random_queries(inserted, n_query)
near_Q = make_near_queries(inserted, n_query)

print("Theoretical FPR (good hashing):", theoretical_fpr(m, k, n_insert))


Theoretical FPR (good hashing): 0.1377817072916896


### This cell defines all the helper functions, BloomFilter class,
### and generates the input data (inserted items + queries)
### that will be used in the experiments.

# Hash Function Families (Good vs. Flawed)

This cell defines different hash function behaviors used in our Bloom filter experiments.  
Each function models one of the four qualities we are testing:


In [ ]:

def good_hash(i, key_bytes):
    return sha256_int(i.to_bytes(4, 'big') + key_bytes)

def high_collision_hash(i, key_bytes):
    return sha256_int(i.to_bytes(4, 'big') + key_bytes) & 0xFFF

def dependent_hash(i, key_bytes):
    base = sha256_int(b'base' + key_bytes)
    return base + i * 13

def make_bad_distribution_hash(m):
    def bad_dist(i, key_bytes):
        base = sha256_int(i.to_bytes(4,'big') + key_bytes)
        return base % (m // 4)
    return bad_dist

def poor_avalanche_hash(i, key_bytes):
    s = sum(key_bytes) + i * 257
    s ^= (s << 7) & 0xFFFFFFFF
    s ^= (s >> 3)
    return s & 0x7FFFFFFF

- **good_hash** → baseline: strong avalanche, independent, uniform distribution.  
- **high_collision_hash** → (a) tests the effect of a *high collision rate*.  
- **dependent_hash** → (b) tests *lack of independence* between hash functions.  
- **make_bad_distribution_hash** → (c) tests *uneven distribution* of bits.  
- **poor_avalanche_hash** → (d) tests *poor avalanche effect* (small input changes → small hash changes).


## (a) Collision Rate — 5 points
Run with good vs high-collision hash families.

In [ ]:
# Run experiment using the good (baseline) hash family
fpr_good, occ_good = run_experiment(m, k, inserted, rand_Q, good_hash)

# Run experiment using the high-collision hash family to test effect of collisions
fpr_coll, occ_coll = run_experiment(m, k, inserted, rand_Q, high_collision_hash)

# Print results for the good hash function
print("Good:", fpr_good, "Occupancy:", occ_good)

# Print results for the high-collision hash function
print("High-collision:", fpr_coll, "Occupancy:", occ_coll)

Good: 0.13585 Occupancy: 0.753
High-collision: 0.78285 Occupancy: 0.3955


### (a) Collision Rate — Insights

- **Good hash:** The false positive rate (FPR) was approximately 0.136, which is near the theoretical value (~ 0.138), indicating an acceptable spread of hashed values.   
- **High-collision hash:** The FPR was approximately 0.783, so much higher due to numerous overlapping hash positions.  
- **Occupancy:** dropped from 0.753 to 0.395, indicating fewer unique bits were used (a more extended hash was used).
- **Conclusion:** A high collision rate significantly increases false positives because many elements point to the same position.


## (b) Independence — 5 points
Run with good vs dependent hash families.

In [ ]:
# experiment with the good hash function (baseline)
fpr_good, occ_good = run_experiment(m, k, inserted, rand_Q, good_hash)

# experiment with the dependent (affine) hash function to test non-independence
fpr_dep, occ_dep = run_experiment(m, k, inserted, rand_Q, dependent_hash)
# Show results for the good hash (should be low FPR)
print("Good:", fpr_good, "Occupancy:", occ_good)

# Show results for the dependent hash (FPR expected to increase)
print("Dependent (affine):", fpr_dep, "Occupancy:", occ_dep)

Good: 0.13585 Occupancy: 0.753
Dependent (affine): 0.49565 Occupancy: 0.756


### (b) Independence — Insights

- With the good hash, FPR is around **0.136**, which is very close to what theory says.  
- But with the dependent hash (where all hash functions kinda follow each other), FPR jumps to **0.496**.  
- Bit occupancy stayed similar (~0.75) in both, so the issue isn't about fullness of the filter.  
- This shows that when hashes aren't independent, it messes up the spreading and makes false positives a lot worse.


## (c) Distribution — 5 points
Run with good vs biased distribution hash families.

In [ ]:
# Create a biased hash function that only uses the first quarter of the bit array
bad_dist = make_bad_distribution_hash(m)

# Run experiment using the good hash function (baseline, uniform distribution)
fpr_good, occ_good = run_experiment(m, k, inserted, rand_Q, good_hash)

# experiment using the biased hash function (non-uniform distribution)
fpr_bias, occ_bias = run_experiment(m, k, inserted, rand_Q, bad_dist)

# Print results for both hash functions
print("Good:", fpr_good, "Occupancy:", occ_good)
print("Biased:", fpr_bias, "Occupancy:", occ_bias)

Good: 0.13585 Occupancy: 0.753
Biased: 0.97785 Occupancy: 0.2492


### (c) Distribution — Insights

- The good hash shows a low FPR of **0.136**, meaning the filter works normally.  
- The biased hash shoots up to **0.978**, almost always giving false positives.  
- Occupancy dropped from **0.753 → 0.249**, showing only a small part of the filter got used.  
- So yeah, bad distribution totally ruins the filter — bits get crowded in one area, and everything looks like it’s already there.


## (d) Avalanche Effect — 5 points
Run with good vs poor-avalanche hash families (using near queries).

In [ ]:
# experiment using the good hash function with near queries (1 character difference)
fpr_good, occ_good = run_experiment(m, k, inserted, near_Q, good_hash)

# using the poor-avalanche hash function with near queries
fpr_poor, occ_poor = run_experiment(m, k, inserted, near_Q, poor_avalanche_hash)

# Print results for the good hash (should stay low even with near queries)
print("Good (near queries):", fpr_good, "Occupancy:", occ_good)
# Print results for poor avalanche hash (expected to give high FPR)
print("Poor avalanche:", fpr_poor, "Occupancy:", occ_poor)

Good (near queries): 0.14355 Occupancy: 0.753
Poor avalanche: 0.97075 Occupancy: 0.1717


### (d) Avalanche Effect — Insights

- With the good hash, FPR stays low at **0.144**, even with near queries (just 1-char difference).  
- Poor-avalanche hash jumps to **0.971** FPR — almost always wrong.  
- Occupancy drops to **0.1717**, showing it keeps hitting the same few bits.  
- So yeah, without avalanche, small changes in input barely move the bits, and we get tons of false positives.


# Visualization - Bar Graph


In [ ]:
# Build labels and FPR values
labels = [f"{p}\n{h}" for p,h,_,_ in results]
fprs   = [r[2] for r in results]

# One clean matplotlib chart (no seaborn, no custom colors)
plt.figure(figsize=(10,5))
plt.bar(range(len(fprs)), fprs)
plt.xticks(range(len(fprs)), labels, rotation=45, ha='right')
plt.ylabel("False positive rate")
plt.title("Bloom Filter: Effect of Hash Quality on False Positives")
plt.tight_layout()
plt.show()


NameError: name 'results' is not defined

###  Overall Insights from Bar Chart

- The **good hash** shows consistently low false positive rate (FPR ≈ 0.136) across all tests.
- **High-collision** and **biased distribution** hash families cause FPR to spike close to **0.78–0.97**, meaning the Bloom filter gives incorrect positives most of the time.
- **Dependent hashes** (non-independent) and **poor avalanche** behavior also raise FPR significantly, even though their bit occupancy seems “normal”.
- In short, **all four hash qualities matter** — and bad behavior in any one of them leads to a much worse Bloom filter.
